# 02 · Arrays: Hashing, Prefix Sums & Subarrays

Classic interview problems that build on array basics. Each one shows the **brute force → better → optimal** progression.

**Topics covered**
1. Linear search
2. Union of two sorted arrays
3. Missing number
4. Maximum consecutive ones
5. Two Sum
6. Maximum subarray sum (Kadane's algorithm)
7. Maximum value of a string in an array
8. Best time to buy and sell a stock
9. Rearrange array elements by sign
10. Longest consecutive sequence

## 1. Linear search

Check every element one by one. Return the index of the first match, or `-1` if the target is missing.

**Time:** O(n) · **Space:** O(1)

In [1]:
def linear_search(nums, target):
    for i in range(len(nums)):
        if nums[i] == target:
            return i                     # stop at the first match
    return -1                            # looked at everything, not found

nums = [5, 3, 9, 8, 1, 6, 4, -10, -100]
print(linear_search(nums, 9))            # 2
print(linear_search(nums, 100))          # -1

2
-1


## 2. Union of two sorted arrays

The union contains every **distinct** value that appears in either array.

- **Set approach:** put everything in a `set`, then sort. Time O(n log n).
- **Two-pointer merge:** because both arrays are already sorted, walk through them together like the merge step of merge sort. Time O(n + m).

The trick: before appending a value, check it is different from the **last value added**, so duplicates are skipped.

In [2]:
nums1 = [1, 1, 1, 2, 3, 4, 6, 7]
nums2 = [1, 2, 3, 6, 7, 8, 9, 10]

# Approach 1: set + sort
union = sorted(set(nums1) | set(nums2))
print("Set       :", union)

# Approach 2: two-pointer merge on sorted arrays
def union_sorted(a, b):
    i = j = 0
    result = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            if not result or result[-1] != a[i]:   # skip duplicates
                result.append(a[i])
            i += 1
        else:
            if not result or result[-1] != b[j]:
                result.append(b[j])
            j += 1
    # one array may still have leftovers
    while i < len(a):
        if not result or result[-1] != a[i]:
            result.append(a[i])
        i += 1
    while j < len(b):
        if not result or result[-1] != b[j]:
            result.append(b[j])
        j += 1
    return result

print("Two-ptr   :", union_sorted(nums1, nums2))
assert union == union_sorted(nums1, nums2)

Set       : [1, 2, 3, 4, 6, 7, 8, 9, 10]
Two-ptr   : [1, 2, 3, 4, 6, 7, 8, 9, 10]


> 🐞 **Fix:** the earlier two-pointer attempt never moved `j` inside the first loop, and its test input `nums1` was not sorted. The two-pointer merge **only works on sorted arrays**.

## 3. Missing number

Array of `n` numbers taken from `0..n`, exactly one is missing.

| Approach | Idea | Time | Space |
|---|---|---|---|
| Brute | for every `i` in `0..n`, check `i not in nums` | O(n²) | O(1) |
| **Sum formula** | expected sum `n(n+1)/2` minus actual sum | O(n) | O(1) |
| XOR | `a ^ a = 0`, so XOR of all indices and values leaves the missing one | O(n) | O(1) |

Use `//` (integer division) so the answer stays an `int` instead of `2.0`.

In [3]:
nums = [0, 1, 3, 4]
n = len(nums)

# Brute force
for i in range(n + 1):
    if i not in nums:
        print("Brute:", i)

# Sum formula
expected = n * (n + 1) // 2              # sum of 0..n
actual = sum(nums)
print("Sum  :", expected - actual)       # (renamed: don't shadow built-in sum)

# XOR trick
x = 0
for i in range(n + 1):
    x ^= i                               # XOR all numbers 0..n
for v in nums:
    x ^= v                               # XOR again with the array → pairs cancel
print("XOR  :", x)

Brute: 2
Sum  : 2
XOR  : 2


## 4. Maximum consecutive ones

Count the current run of `1`s; whenever a `0` appears, the run ends so we compare it to the best seen and reset.

**Time:** O(n) · **Space:** O(1)

In [4]:
nums = [1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1]

# Version 1: store every run, then take the max
runs, count = [], 0
for x in nums:
    if x == 1:
        count += 1
    else:
        runs.append(count)
        count = 0
runs.append(count)                       # the last run has no 0 after it!
print("Runs:", runs, "→ max =", max(runs))

# Version 2: no extra list needed
best = count = 0
for x in nums:
    count = count + 1 if x == 1 else 0
    best = max(best, count)
print("Max :", best)

Runs: [2, 1, 4, 2] → max = 4
Max : 4


## 5. Two Sum

Find two numbers whose sum equals `target`.

- **Brute force:** try every pair `(i, j)` with `j > i` → O(n²).
- **Hashing:** for each number, we need `target - number`. Keep a dict `value → index` of numbers already seen. One pass → O(n).

> 🐞 **Fix:** the brute-force loop started `j` at 0, which pairs an element with itself and prints each pair twice. Starting `j` at `i + 1` fixes both.

In [5]:
nums = [1, 2, 3, 6, 7, 8, 9, 10]
target = 15

# Brute force
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        if nums[i] + nums[j] == target:
            print("Brute pair at indices:", i, j)

Brute pair at indices: 3 6
Brute pair at indices: 4 5


In [6]:
def two_sum(nums, target):
    seen = {}                            # value → index
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:                 # the partner appeared earlier
            return [seen[need], i]
        seen[x] = i
    return [-1, -1]                      # no pair

print(two_sum([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 15))   # [6, 7] → 7 + 8
print(two_sum([1, 2, 3], 100))                        # [-1, -1]

[6, 7]
[-1, -1]


## 6. Maximum subarray sum

Find the contiguous subarray with the largest sum.

| Approach | Idea | Time |
|---|---|---|
| Brute (3 loops) | recompute each subarray sum | O(n³) |
| Better | fix start `i`, grow end `j`, keep a running sum | O(n²) |
| **Kadane's algorithm** | running sum; if it drops below 0, throw it away and restart | **O(n)** |

**Why Kadane works:** a negative running sum can only *hurt* any subarray that extends it, so we reset it to 0.

In [7]:
nums = [1, 4, -3, 2, 1, 5, 6, -7, 9]
n = len(nums)

# Better: O(n²) with a running sum
best = float("-inf")
for i in range(n):
    current = 0
    for j in range(i, n):
        current += nums[j]               # extend the subarray by one element
        best = max(best, current)
print("O(n²):", best)

O(n²): 18


In [8]:
# Kadane's algorithm
best = float("-inf")
current = 0
for x in nums:
    current += x
    best = max(best, current)            # record BEFORE resetting
    if current < 0:
        current = 0                      # a negative prefix is never worth keeping
print("Kadane:", best)

Kadane: 18


In [9]:
# Kadane + the actual subarray (track where the current run started)
best = float("-inf")
current = 0
start = 0
best_start = best_end = 0
for i, x in enumerate(nums):
    if current == 0:
        start = i                        # a fresh run begins here
    current += x
    if current > best:
        best = current
        best_start, best_end = start, i
    if current < 0:
        current = 0
print("Max sum:", best, "| subarray:", nums[best_start:best_end + 1], "| indices:", (best_start, best_end))

Max sum: 18 | subarray: [1, 4, -3, 2, 1, 5, 6, -7, 9] | indices: (0, 8)


> 🐞 **Fix:** the earlier O(n²) cell had a typo (`curren_sum` vs `current_sum`) and never appended to the list, so it crashed. Tracking `best` directly avoids needing a list of all sums.

## 7. Maximum value of a string in an array

For each string: if it contains **only digits**, its value is the number it represents; otherwise its value is its **length**. Return the maximum value.

`str.isdigit()` tells us whether every character is a digit.

In [10]:
strs = ["alic3", "bob", "3", "4", "00000"]
max_value = 0
for s in strs:
    value = int(s) if s.isdigit() else len(s)
    max_value = max(max_value, value)
print("Maximum value:", max_value)       # "alic3" → 5 (letters present → use length)

Maximum value: 5


## 8. Best time to buy and sell a stock

Buy on one day, sell on a **later** day, maximise profit.

- **Brute force:** try every (buy, sell) pair → O(n²).
- **One pass:** keep the **cheapest price so far**. At each day, the best profit if we sell today = `price - min_price_so_far`.

**Time:** O(n) · **Space:** O(1)

In [11]:
prices = [7, 2, 1, 5, 6, 4, 8]

# Brute force
max_profit = 0
for i in range(len(prices)):
    for j in range(i + 1, len(prices)):
        max_profit = max(max_profit, prices[j] - prices[i])
print("Brute:", max_profit)

# One pass
def max_profit_fn(prices):
    min_price = float("inf")
    best = 0
    for price in prices:
        min_price = min(min_price, price)        # cheapest day to buy so far
        best = max(best, price - min_price)      # profit if we sell today
    return best

print("Optimal:", max_profit_fn(prices))             # 7  (buy 1, sell 8)
print("Another:", max_profit_fn([7, 1, 5, 3, 6, 4])) # 5  (buy 1, sell 6)

Brute: 7
Optimal: 7
Another: 5


## 9. Rearrange array elements by sign

Given equal numbers of positives and negatives, rearrange so signs **alternate**, starting with a positive, while keeping the relative order inside each sign group.

- **Brute:** split into `pos` and `neg` lists, then interleave. (`pop(0)` is O(n); using indexes is cleaner.)
- **Optimal:** one pass, write positives to even indexes (0, 2, 4…) and negatives to odd indexes (1, 3, 5…).

In [12]:
nums = [5, 10, -3, -1, -10, 6]
n = len(nums)

# Brute force: two lists, then interleave
pos = [x for x in nums if x >= 0]
neg = [x for x in nums if x < 0]
result = []
for i in range(n // 2):
    result.append(pos[i])
    result.append(neg[i])
print("Two lists:", result)

# Optimal: place directly into the answer array
result = [0] * n
pos_index, neg_index = 0, 1
for x in nums:
    if x >= 0:
        result[pos_index] = x
        pos_index += 2                   # next even slot
    else:
        result[neg_index] = x
        neg_index += 2                   # next odd slot
print("One pass :", result)

Two lists: [5, -3, 10, -1, 6, -10]
One pass : [5, -3, 10, -1, 6, -10]


## 10. Longest consecutive sequence

Find the length of the longest run of values like `4, 5, 6, 7` (order in the input does **not** matter).

| Approach | Idea | Time |
|---|---|---|
| Sort | sort, then count runs (skip duplicates) | O(n log n) |
| **Set** | only start counting at numbers with **no predecessor** (`num-1` not in set) | **O(n)** |

In the set approach each number is visited at most twice (once as an element, once while extending a run), so it stays linear.

In [13]:
nums = [4, 5, 6, 7, 1, 3, 8, 34, 56, 56, 57, 58, 89, 59, 60, 61, 62]

# Approach 1: sort
arr = sorted(nums)
last_smaller = float("-inf")
count = longest = 0
for num in arr:
    if num - 1 == last_smaller:          # continues the current run
        count += 1
        last_smaller = num
    elif num != last_smaller:            # a gap → start a new run (equal values are skipped)
        count = 1
        last_smaller = num
    longest = max(longest, count)
print("Sorting:", longest)

# Approach 2: set
seen = set(nums)
longest = 0
for num in seen:
    if num - 1 not in seen:              # `num` is the START of a run
        x, count = num, 1
        while x + 1 in seen:
            x += 1
            count += 1
        longest = max(longest, count)
print("Set    :", longest)               # 56..62 → 7

Sorting: 7
Set    : 7


---
### ✅ Key takeaways
- A **dict / set** turns "search for a partner" from O(n) into O(1) → Two Sum, longest consecutive sequence.
- **Kadane's algorithm**: reset the running sum when it goes negative.
- **One pass with "best so far"** variables (min price, running max) shows up everywhere.